Steps for k nearest neighbors:
1. Load dataset
2. Split into train/test splits (90% train, 10% test)
3. Distance function
4. Implement train/test functions
5. Optimize hyperparams (k, L1 or L2 distance)

In [63]:
import pandas as pd
import numpy as np

In [64]:
df = pd.read_csv('iris.csv')

df_train = df.sample(frac=0.9, random_state=42)
df_test = df.drop(df_train.index)

X_train = df_train.iloc[:, :-1].to_numpy()
y_train = df_train.iloc[:, -1].to_numpy()

X_test = df_test.iloc[:, :-1].to_numpy()
y_test = df_test.iloc[:, -1].to_numpy()

print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Training data shape:  (134, 4)
Training labels shape:  (134,)
Test data shape:  (15, 4)
Test labels shape:  (15,)


In [65]:
class kNearestNeighbors:
    def __init__(self, k, dist_type="l2"):
        self.k = k
        self.dist_type = dist_type
        self.X_train = None
        self.y_train = None

    def train(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def predict(self, X_test):
        preds = []

        for row in X_test:
            neighs = self.get_neighbors(row)
            pred = max(set(neighs), key=neighs.count)
            preds.append(pred)

        return np.array(preds)

    def get_neighbors(self, row):
        dists = []

        for i, train_row in enumerate(self.X_train):
            if self.dist_type == "l1":
                dist = self.l1_distance(row, train_row)
            else:
                dist = self.l2_distance(row, train_row)

            dists.append((dist, i))
        
        dists.sort()
        neighs = []
        for i in range(self.k):
            neighs.append(self.y_train[dists[i][1]])
        return neighs

    def l1_distance(self, row1, row2):
        diff = row1 - row2
        dist = np.sum(np.abs(diff))
        return dist  

    def l2_distance(self, row1, row2):
        diff = row1 - row2
        diff_sq = np.square(diff)
        sum_sq = np.sum(diff_sq)
        dist = np.sqrt(sum_sq)
        return dist

In [69]:
k = 5
dist_type = "l2"
num_test = len(y_test)

classifier = kNearestNeighbors(k, dist_type)
classifier.train(X_train, y_train)
preds = classifier.predict(X_test)

num_correct = np.sum(preds == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

Got 14 / 15 correct => accuracy: 0.933333


In [67]:
# param sweep
best_model = None
best_correct = 0

for dist_type in ["l1", "l2"]:
    for k in range(1, 10, 2):
        classifier = kNearestNeighbors(k, dist_type)
        classifier.train(X_train, y_train)
        preds = classifier.predict(X_test)

        num_correct = np.sum(preds == y_test)

        if num_correct > best_correct:
            best_model = (k, dist_type)
            best_correct = num_correct

accuracy = float(best_correct) / num_test
print(f"Best model: k={best_model[0]}, dist={best_model[1]}, accuracy={accuracy}")

Best model: k=1, dist=l1, accuracy=0.9333333333333333
